In [0]:
# CELL 1: Setup & Load Silver
from pyspark.sql.functions import col, sum, avg, count, month, year, round, desc

storage_account_name = "REMOVED FOR SECURITY"
storage_account_key = "REMOVED FOR SECURITY"
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

print("Loading Silver Data...")
df_silver_trans = spark.read.format("delta").load(f"{silver_path}/transactions")
df_silver_units = spark.read.format("delta").load(f"{silver_path}/units")

print("✅ Silver Data Loaded & Gold Container Configured.")

In [0]:
# CELL 2: Gold Table 1 - Monthly Sales Trends
gold_sales_overview = df_silver_trans.groupBy(
    year("transaction_date").alias("year"),
    month("transaction_date").alias("month")
).agg(
    sum("price").alias("total_sales_aed"),
    count("transaction_id").alias("total_transactions"),
    avg("price_per_sqft").alias("avg_price_sqft")
).orderBy("year", "month")

gold_sales_overview = gold_sales_overview.withColumn("total_sales_aed", round("total_sales_aed", 2))
gold_sales_overview = gold_sales_overview.withColumn("avg_price_sqft", round("avg_price_sqft", 2))

print("✅ Sales Overview Created.")
display(gold_sales_overview)

In [0]:
# CELL 3: Gold Table 2 - Area Performance
gold_area_stats = df_silver_trans.groupBy("area_name").agg(
    sum("price").alias("total_volume_aed"),
    count("transaction_id").alias("transaction_count"),
    avg("price_per_sqft").alias("avg_sqft_price")
).orderBy(desc("total_volume_aed"))

gold_area_stats = gold_area_stats.withColumn("total_volume_aed", round("total_volume_aed", 0))
gold_area_stats = gold_area_stats.withColumn("avg_sqft_price", round("avg_sqft_price", 2))

print("✅ Area Stats Created.")
display(gold_area_stats.limit(10))

In [0]:
# CELL 4: Gold Table 3 - Supply & Inventory
gold_supply_stats = df_silver_units.groupBy(
    "project_number",
    "room_type",
    "bedrooms"
).agg(
    count("unit_number").alias("unit_count"),
    avg("unit_area").alias("avg_unit_area")
).orderBy(desc("unit_count"))

print("✅ Supply Stats Created.")
display(gold_supply_stats.limit(10))

In [0]:
# CELL 5: Write to Gold Layer
print("Writing Gold Tables...")

gold_sales_overview.write.format("delta").mode("overwrite").save(f"{gold_path}/sales_overview")
gold_area_stats.write.format("delta").mode("overwrite").save(f"{gold_path}/area_stats")
gold_supply_stats.write.format("delta").mode("overwrite").save(f"{gold_path}/supply_stats")

print("🎉 SUCCESS: Gold Layer (Analytics Ready) Created!")

In [0]:
# CELL 6: Verification
try:
    check_df = spark.read.format("delta").load(f"{gold_path}/sales_overview")
    count = check_df.count()

    print(f"✅ VERIFIED: Gold Sales Table contains {count} monthly records.")

    if count > 0:
        display(check_df.limit(5))
    else:
        print("⚠️ WARNING: Table is empty.")

except Exception as e:
    print(f"❌ ERROR: {e}")